# Tree comparison: how much do the choices matter?

Executed once by the `tree_comparison` rule, after all trees are built. The trees differ in two choices:

| choice | values | meaning |
|---|---|---|
| `derep` | `all` / `derep` | every genome, or only the dereplicated representatives (external genomes always kept) |
| `copies` | `single` / `multi` | Complete BUSCO genes only, or also the top-scoring copy of Duplicated genes |

Trees are labelled `derep/copies`, e.g. `all/multi`. A quick-check subset run compares only subset
trees; do not compare its numbers with a full run.

Sections: (1) topological distance between the trees, (2) clade fragmentation side by side,
(3) single- vs multi-copy: does adding the Duplicated genes make genes better or worse at separating clades?

## 0. Setup

Parameters come from the JSON file named by `NB_PARAMS`, written by the Snakemake rule.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
from Bio import Phylo

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_accuracy_lib as lib
import tree_diagnostics_lib as diag

OUT = Path(PARAMS["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)

In [ ]:
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])

# Only trees that exist (a variant may have failed or not been requested)
trees = {entry["label"]: entry for entry in PARAMS["trees"] if Path(entry["tree"]).exists()}
loaded = {label: Phylo.read(entry["tree"], "newick") for label, entry in trees.items()}
labels = list(loaded)

show_all(pl.DataFrame({"tree": labels, "tips": [loaded[label].count_terminals() for label in labels]}))

## 1. Topological distance

Normalised **Robinson-Foulds** distance on the tips two trees share: the share of branches (bipartitions)
found in one tree but not the other. 0 = identical topology, 1 = nothing in common.

**How to read it.** Compare the distance caused by one choice with the distance caused by another.
If `single` vs `multi` (same genomes) is much smaller than `all` vs `derep` (different genomes), the
Duplicated genes barely change the tree and the genome set matters more. Distances of a few percent are
what you would expect from tree-search noise alone, so only large differences are meaningful.

In [ ]:
rf = np.zeros((len(labels), len(labels)))
for i, first in enumerate(labels):
    for j, second in enumerate(labels):
        if i != j:
            rf[i, j] = lib.rf_distance(loaded[first], loaded[second])[1]

rf_table = pl.DataFrame({"tree": labels, **{label: np.round(rf[:, k], 3) for k, label in enumerate(labels)}})
show_all(rf_table)
rf_table.write_csv(OUT / "rf_matrix.tsv", separator="\t")

if len(labels) > 1:
    fig, axis = plt.subplots(figsize=(5, 4))
    image = axis.imshow(rf, vmin=0)
    axis.set_xticks(range(len(labels)))
    axis.set_xticklabels(labels, rotation=45, ha="right")
    axis.set_yticks(range(len(labels)))
    axis.set_yticklabels(labels)
    plt.colorbar(image)
    plt.show()

## 2. Clade fragmentation by tree

Same measure as in the `tree_accuracy` notebook: a database name is split into several *pieces* when its
tips do not form one clade (unrooted, unlabelled tips ignored). `extra_pieces` is the number of pieces
beyond one per name; `fragmented_names` is the number of names affected.

**How to read it.** Trees with fewer genomes (`derep`) have fewer opportunities to fragment, so a drop in
fragmentation there is not by itself evidence of a better tree. The more informative comparison is
`single` vs `multi` on the same genomes. The last table lists the names whose piece count changes between
trees, which shows exactly which clades the genome set or the gene set affects.

In [ ]:
rows, pieces_by_tree = [], {}
for label, tree in loaded.items():
    summary, names = lib.fragmentation(tree, taxonomy)
    rows += [{"tree": label, **row} for row in summary]
    pieces_by_tree[label] = {(n["rank"], n["name"]): n["pieces"] for n in names}

by_tree = pl.DataFrame(rows)
by_tree.write_csv(OUT / "fragmentation_by_tree.tsv", separator="\t")
show_all(by_tree.pivot(on="rank", index="tree", values="extra_pieces").select(["tree", *lib.RANKS]))
print("(extra pieces per rank)")

In [ ]:
# Names whose number of pieces differs between trees
all_names = sorted(set().union(*[set(counts) for counts in pieces_by_tree.values()]))
changed = pl.DataFrame([
    {"rank": rank, "name": name, **{label: pieces_by_tree[label].get((rank, name), 0) for label in labels}}
    for rank, name in all_names
    if len({pieces_by_tree[label].get((rank, name), 0) for label in labels}) > 1
])
print(f"{changed.height} names have a different number of pieces in at least one tree")
if changed.height:
    spread = changed.select(labels).max_horizontal() - changed.select(labels).min_horizontal()
    show_all(changed.with_columns(spread.alias("spread")).sort("spread", descending=True).head(15).drop("spread"))

## 3. Single vs multi copy

Does adding the top copy of Duplicated genes help? For each `derep` value the same genomes are used in the
`single` and `multi` alignments, and **each gene is scored on its own**:

`separation` = probability that a within-clade pair of genomes is closer (p-distance, that gene only) than
a between-clade pair. 1 = the gene separates the clades perfectly, 0.5 = no information.

Columns: `median_single` / `median_multi` - median over genes; `multi_better` - share of genes whose
separation is higher with the extra copies; `pair_separation_*` - the same on the concatenated alignment.

**How to read it.** Differences in the third decimal are noise. A clearly higher `median_multi` and
`multi_better` well above 50% would say the extra copies add real signal; a lower value would say the
top-score rule is adding paralogs. See the multicopy section of `tree_diagnostics` for how often the chosen
copy is the peer-like one.

In [ ]:
RANKS_COMPARED = ["order", "family"]
N_PAIRS = PARAMS["n_pairs"]


def gene_separation_for(tree_entry):
    '''
    Per-gene and overall separation of one alignment at the compared ranks.

    Parameters
    ----------
    tree_entry : dict
        One entry of PARAMS["trees"] (alignment and partitions paths).

    Returns
    -------
    dict
        {rank: (pl.DataFrame[gene, separation], overall pair separation)}
    '''
    ids, matrix = diag.load_alignment(tree_entry["alignment"])
    partitions = diag.read_partitions(tree_entry["partitions"])
    _, present = diag.occupancy(ids, matrix, partitions)
    report = diag.rank_report(ids, matrix, present, partitions, taxonomy,
                              ranks=RANKS_COMPARED, n_pairs=N_PAIRS, seed=3)
    return {
        rank: (pl.DataFrame(diag.gene_separation(report, partitions, rank),
                            schema=["gene", "separation", "pairs"], orient="row"),
               report[rank]["summary"]["separation_pdist"])
        for rank in report
    }

In [ ]:
results = []
for derep in sorted({entry["derep"] for entry in trees.values()}):
    single, multi = trees.get(f"{derep}/single"), trees.get(f"{derep}/multi")
    if not single or not multi or not (Path(single["alignment"]).exists() and Path(multi["alignment"]).exists()):
        print(f"{derep}: needs both a single and a multi alignment, skipped")
        continue
    sep_single, sep_multi = gene_separation_for(single), gene_separation_for(multi)
    for rank in RANKS_COMPARED:
        genes = (sep_single[rank][0].join(sep_multi[rank][0], on="gene", suffix="_multi")
                 .drop_nulls().filter(pl.col("separation").is_not_nan() & pl.col("separation_multi").is_not_nan()))
        genes.select("gene", pl.col("separation").alias("single"), pl.col("separation_multi").alias("multi")) \
             .write_csv(OUT / f"gene_separation_{derep}_{rank}.tsv", separator="\t")
        results.append({
            "derep": derep, "rank": rank, "genes": genes.height,
            "median_single": round(genes["separation"].median(), 3),
            "median_multi": round(genes["separation_multi"].median(), 3),
            "multi_better": round(float((genes["separation_multi"] > genes["separation"]).mean()), 3),
            "pair_separation_single": round(float(sep_single[rank][1]), 3),
            "pair_separation_multi": round(float(sep_multi[rank][1]), 3),
        })
show_all(pl.DataFrame(results) if results else pl.DataFrame({"note": ["no single/multi pair available"]}))

## 4. Summary

Tables written next to this notebook: `rf_matrix.tsv`, `fragmentation_by_tree.tsv` and
`gene_separation_<derep>_<rank>.tsv`.

In [ ]:
print("Trees compared:", ", ".join(labels))
if len(labels) > 1:
    pairs = [(a, b, rf[i, j]) for i, a in enumerate(labels) for j, b in enumerate(labels) if i < j]
    nearest, farthest = min(pairs, key=lambda p: p[2]), max(pairs, key=lambda p: p[2])
    print(f"Most similar : {nearest[0]} vs {nearest[1]} (RF {nearest[2]:.3f})")
    print(f"Most different: {farthest[0]} vs {farthest[1]} (RF {farthest[2]:.3f})")
for row in results:
    print(f"{row['derep']}/{row['rank']}: multi better for {row['multi_better']:.0%} of genes "
          f"(median {row['median_single']} -> {row['median_multi']})")